In [2]:
from pathlib import Path
import pandas as pd
import anndata as ad
from scipy import sparse
from tqdm import tqdm
import re

In [3]:
def _read_disignatlas_profile(path: Path) -> pd.DataFrame:
    dataset_id = path.stem.replace("_profile", "")

    # IMPORTANT: skip the first line "# count"
    df = pd.read_csv(
        path,
        skiprows=1,
        index_col=0,
        low_memory=False,
    )

    # Clean labels
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()

    # Convert expression values to numeric
    df = df.apply(pd.to_numeric, errors="coerce").fillna(0)

    # Collapse duplicate genes within a file
    if not df.index.is_unique:
        df = df.groupby(level=0, sort=False).sum()

    # Make sample names unique across files
    df.columns = [f"{dataset_id}__{c}" for c in df.columns]

    return df


def load_disignatlas_stream(max_files: int | None = None) -> ad.AnnData:
    disignatlas_path = Path("/cluster/work/boeva/eheiss/datasets/DiSignAtlas/dsa_exp_download")
    files = sorted(disignatlas_path.glob("*_profile.csv"))

    if max_files is not None:
        files = files[:max_files]

    adatas = []
    gene_index = None

    for path in tqdm(files):
        df = _read_disignatlas_profile(path)

        # Use the first file's genes as reference
        if gene_index is None:
            gene_index = df.index
        else:
            df = df.reindex(gene_index, fill_value=0)

        # samples x genes
        X = sparse.csr_matrix(df.T.to_numpy(dtype="float32"))

        adata = ad.AnnData(X)
        adata.var_names = gene_index.astype(str)
        adata.obs_names = pd.Index(df.columns.astype(str))
        adata.obs["dataset"] = path.stem.replace("_profile", "")

        adatas.append(adata)

    out = ad.concat(adatas, axis=0, join="inner", merge="same")
    out.obs_names_make_unique()
    return out

In [4]:
disignatlas = load_disignatlas_stream()

100%|██████████| 9978/9978 [38:51<00:00,  4.28it/s]  


In [6]:
df = pd.read_csv(
    "/cluster/customapps/biomed/boeva/eheiss/downloads/gene2ensembl.gz",
    sep="\t",
    comment="#",
    header=None
)

df.columns = [
    "tax_id", "GeneID", "Ensembl_gene_id",
    "RNA_nucleotide_accession",
    "Ensembl_rna_id",
    "protein_accession",
    "Ensembl_protein_id"
]
df = df[df["tax_id"] == 9606]

In [7]:
mapping = df[["GeneID", "Ensembl_gene_id"]].dropna().copy()

del df

mapping["GeneID"] = mapping["GeneID"].astype(str)

# keep only one Ensembl ID per GeneID
mapping = mapping.drop_duplicates(subset=["GeneID"], keep="first")

In [8]:
# current gene names in DiSignAtlas = Entrez IDs as strings
genes = disignatlas.var_names.astype(str)

# keep original positions explicitly
gene_df = pd.DataFrame({
    "orig_pos": range(len(genes)),
    "GeneID": genes,
})

# mapping must also be string-typed
mapping["GeneID"] = mapping["GeneID"].astype(str)

gene_df = gene_df.merge(mapping, on="GeneID", how="left")

# keep only genes that mapped
mask = gene_df["Ensembl_gene_id"].notna().to_numpy()

# subset with boolean mask, not merged index
disignatlas = disignatlas[:, mask].copy()

# assign mapped Ensembl IDs
mapped_ensembl = gene_df.loc[mask, "Ensembl_gene_id"].astype(str).to_numpy()

# store original Entrez IDs
disignatlas.var["entrez_id"] = disignatlas.var_names.astype(str)

# replace var_names
disignatlas.var_names = mapped_ensembl

# remove duplicated Ensembl IDs
disignatlas = disignatlas[:, ~disignatlas.var_names.duplicated()].copy()

In [11]:
metadata = pd.read_csv(
    "/cluster/work/boeva/eheiss/datasets/DiSignAtlas/Disease_information_Datasets.csv",
    encoding="latin-1",
)
id_to_disease = dict(zip(metadata["dsaid"].astype(str), metadata["disease"].astype(str)))


In [14]:
BULKFORMER_DISEASES = {
    "COVID-19", "Amyotrophic Lateral Sclerosis", "Colorectal Carcinoma",
    "Type 1 Diabetes", "Asthma", "Breast Cancer", "Psoriasis",
    "Alzheimer's Disease", "Parkinson's Disease", "Systemic Lupus Erythematosus",
    "Hepatocellular Carcinoma", "Chronic Obstructive Pulmonary Disease",
    "Sepsis", "Influenza", "Multiple Sclerosis", "Tuberculosis",
    "Rheumatoid Arthritis", "Idiopathic Pulmonary Fibrosis", "Ulcerative Colitis",
    "Crohn's Disease", "Huntington's Disease", "Acute Myeloid Leukemia (Aml-M2)",
    "Schizophrenia",
}

disignatlas.obs["disease"] = disignatlas.obs["dataset"].map(id_to_disease)
disignatlas = disignatlas[disignatlas.obs["disease"].isin(BULKFORMER_DISEASES)].copy()
human_dsaids = set(metadata.loc[metadata["organism"].str.contains("Homo sapien", na=False), "dsaid"].astype(str))
disignatlas = disignatlas[disignatlas.obs["dataset"].isin(human_dsaids)].copy()


print(disignatlas.obs["disease"].value_counts().to_string())
print(f"\nTotal diseases: {disignatlas.obs['disease'].nunique()}")
print(f"Total samples:  {disignatlas.n_obs}")


Asthma                                   5833
Ulcerative Colitis                       5134
Crohn's Disease                          4732
Schizophrenia                            4483
Systemic Lupus Erythematosus             4151
COVID-19                                 3600
Chronic Obstructive Pulmonary Disease    3561
Hepatocellular Carcinoma                 3339
Alzheimer's Disease                      2762
Breast Cancer                            2627
Colorectal Carcinoma                     2526
Psoriasis                                2526
Parkinson's Disease                      2491
Multiple Sclerosis                       2400
Idiopathic Pulmonary Fibrosis            2064
Rheumatoid Arthritis                     1995
Type 1 Diabetes                          1967
Tuberculosis                             1945
Huntington's Disease                     1668
Sepsis                                   1327
Influenza                                1289
Acute Myeloid Leukemia (Aml-M2)   

In [15]:
disignatlas.write("/cluster/work/boeva/eheiss/datasets/DiSignAtlas/disignatlas.h5ad")